In [2]:
import matplotlib.pyplot as pyplot
import pandas as pd
import numpy as np
import requests
from google.colab import drive

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
temp_wind = pd.read_csv("/content/drive/MyDrive/Nino/raw_data/era5_temp_wind.csv")
temp_wind

,GEOID,date,temperature_2m,dewpoint_temperature_2m,u_component_of_wind_10m,v_component_of_wind_10m
0,29510,1950-02,275.626884,269.373845,0.498579,-0.060322
1,32510,1950-02,273.741139,269.897285,0.682690,0.492383
2,51580,1950-02,274.914193,269.254715,1.018243,-0.098770
3,51530,1950-02,275.653979,269.529050,0.870680,-0.055304
4,51678,1950-02,275.521934,269.349241,0.882494,-0.036135
...,...,...,...,...,...,...
2850031,44003,2026-06,294.335452,286.482536,1.411837,0.505204
2850032,9007,2026-06,294.259586,287.046827,1.377821,0.419829
2850033,9013,2026-06,294.134139,285.990074,1.257948,0.147022
2850034,9003,2026-06,294.414760,286.620736,1.199917,0.045587


In [5]:
temp_wind["wind_speed"] = np.sqrt(temp_wind["u_component_of_wind_10m"]**2 + temp_wind["v_component_of_wind_10m"]**2)
temp_wind["GEOID"] = temp_wind["GEOID"].astype(str).str.zfill(5)

In [6]:
# create the temperature anomaly, comparing each county-month's temperature against that county and that month's average
temp_wind["cal_month"] = temp_wind["date"].str[5:7]
for col in ["temperature_2m",	"dewpoint_temperature_2m"]:
  # groups values by their county and month pairs and gets the mean of the given col for each county and each month
  clim = temp_wind[temp_wind["date"].between("1996-01", "2026-06")].groupby(["GEOID", "cal_month"])[col].mean().rename(col+"_normal").reset_index()
  temp_wind = temp_wind.merge(clim, on=["GEOID", "cal_month"], how="left")
  temp_wind[col+"_anomaly"] = temp_wind[col] - temp_wind[col+"_normal"]

In [7]:
temp_wind.drop(columns=["temperature_2m",	"dewpoint_temperature_2m",	"u_component_of_wind_10m",	"v_component_of_wind_10m",	"cal_month",	"temperature_2m_normal", "dewpoint_temperature_2m_normal"], inplace=True)

In [26]:
# used to compare these computed anomalies vs realtime anomalies generated in realtime_data.ipynb

# temp_wind["month"] = temp_wind["date"].astype(str).str[5:]
# temp_wind["year"] = temp_wind["date"].astype(str).str[0:4]

# temp_wind_new = temp_wind[(temp_wind["year"].astype(int) >= 2025) & (temp_wind["month"] == "07")]

# print(temp_wind_new["dewpoint_temperature_2m_anomaly"].describe())

count    3107.000000
mean        1.418376
std         1.207050
min        -4.918521
25%         0.760291
50%         1.553891
75%         2.207129
max         3.706695
Name: dewpoint_temperature_2m_anomaly, dtype: float64


In [9]:
temp_wind.to_parquet("/content/drive/MyDrive/Nino/intermediate_merged_data/temp_wind_dynamic.parquet")

In [ ]:
# create the one-time anomaly dataset needed for the realtime_data notebook
# temp_wind = temp_wind.drop(columns=["date", "temperature_2m", "dewpoint_temperature_2m", "u_component_of_wind_10m", "v_component_of_wind_10m", "wind_speed",	"temperature_2m_anomaly","dewpoint_temperature_2m_anomaly"])
# temp_wind = temp_wind.drop_duplicates(subset=["GEOID", "cal_month"])
# temp_wind.to_parquet("/content/drive/MyDrive/Nino/cleaned_data/temp_humid_anomalies.parquet")